# Simple pulse train (multi-view ECG)

This notebook mirrors `examples/01_simple_pulse.py` and is meant for interactive exploration.
It builds a pulse-train process, defines three ECG-style views, generates a small batch, and plots two views.

Requirements:
- `torch` and `toyts`
- `matplotlib` for plotting (skip the plotting cell if you do not have it installed)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from toyts.core.pipeline import SynthPipeline
from toyts.core.utils import utils_make_canonical_A0
from toyts.processes.pulse_train import PulseTrainProcess
from toyts.views.ecg_leads import ECGLeadsView
from toyts.views.noise import BaselineWanderView, NoiseView, NormalizeView
from toyts.views.sampling import SamplingAggregationView


## Device and reproducibility

Use GPU if available and fix the random seed for repeatable results.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rng = torch.Generator(device=device).manual_seed(1234)


## Define the latent process

This process generates a latent PQRST-style pulse train.


In [ ]:
process = PulseTrainProcess(
    seq_len=1024,
    num_pulses=10,
    rhythm_classes=["regular", "irregular", "missed_beat"],
    shape_classes=["gaussian", "sharp_laplace", "biphasic_dog"],
    latent_mode="pqrst3",
    amplitude=2.0,
)


## Define observed views

Views transform the latent process into observed signals.


In [ ]:
A0 = utils_make_canonical_A0(num_leads=8, num_latent=3)  # [C, K]

views = {
    "clean": ECGLeadsView(A0=A0, jitter_std=0.0, max_delay=0),
    "noisy": torch.nn.Sequential(
        ECGLeadsView(A0=A0, jitter_std=0.05, max_delay=3),
        NoiseView(noise_std=0.1),
        BaselineWanderView(amplitude_std=0.3, freq_min=0.05, freq_max=0.2),
    ),
    "normalized_and_resampled": torch.nn.Sequential(
        ECGLeadsView(A0=A0, jitter_std=0.05, max_delay=3),
        NoiseView(noise_std=0.1),
        BaselineWanderView(amplitude_std=0.3, freq_min=0.05, freq_max=0.2),
        NormalizeView(),
        SamplingAggregationView(mode="mean", window=4),
    ),
}


## Run the pipeline and inspect outputs


In [ ]:
pipeline = SynthPipeline(process=process, views=views)
pipeline.to(device)

batch = pipeline(batch_size=4, device=device, rng=rng)

print("Generated batch keys:", batch.keys())
for name, obs in batch.items():
    signal = obs.x  # [B, C, L]
    shape_labels = obs.y["shape"]  # [B]
    rhythm_labels = obs.y["rhythm"]  # [B]

    print(f"--- View: {name} ---")
    print("  Signal shape:", signal.shape)
    print("  Shape labels:", shape_labels)
    print("  Rhythm labels:", rhythm_labels)


## Plot an example

This cell plots one lead from the clean and noisy views and saves the figure under `examples/figures`.


In [ ]:
output_dir = Path("figures")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "01_simple_pulse.png"

clean_signal = batch["clean"].x[0, 0, :].detach().cpu()  # [L]
noisy_signal = batch["noisy"].x[0, 0, :].detach().cpu()  # [L]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
ax1.plot(clean_signal.numpy())
ax1.set_title("View: 'clean'")
ax1.grid(True)
ax2.plot(noisy_signal.numpy())
ax2.set_title("View: 'noisy'")
ax2.grid(True)
plt.tight_layout()
fig.savefig(output_path, dpi=150, bbox_inches="tight")
plt.show()

print("saved figure", output_path)
